# 04 — Modélisation Escalade

**Objectif** : Entraîner un modèle de prédiction du niveau d'escalade.

**Décision de conception** : Le modèle prédit une **classe de niveau** (5 niveaux) plutôt qu'un grade précis.  
Raison : les features disponibles (morpho + années pratique) n'ont pas assez de pouvoir prédictif pour un grade précis (MAE > 6 grades en régression). Une classification par niveau est plus honnête et plus utile.

**Architecture des prédictions :**
- `years_cl = 0` → **Mode POTENTIEL** : règles physiologiques (dead hang, BMI, tractions)
- `years_cl ≥ 1` → **Mode ML** : classification Random Forest → classe de niveau

**Classes :**
| Classe | Grades | Cotation |
|---|---|---|
| 0 — Débutant | < 37 | < 6a |
| 1 — Intermédiaire bas | 37–44 | 6a → 6b+ |
| 2 — Intermédiaire | 45–52 | 6c → 7a+ |
| 3 — Avancé | 53–60 | 7b → 7c+ |
| 4 — Expert | 61+ | 8a et + |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
RANDOM_STATE = 42

## 1. Chargement et préparation

In [ ]:
df = pd.read_csv('../data/processed/climbing_clean.csv')
grades_raw = pd.read_csv('../data/raw/climbing/grades_conversion_table.csv')
grade_map = dict(zip(grades_raw['grade_id'], grades_raw['grade_fra']))
grade_map = {k: v for k, v in grade_map.items() if v != '-'}

if 'bmi' not in df.columns:
    df['bmi'] = df['weight'] / (df['height'] / 100) ** 2

print(f"Dataset : {len(df):,} grimpeurs")

## 2. Création des classes de niveau

In [ ]:
LEVEL_BINS = [0, 37, 45, 53, 61, 85]
LEVEL_LABELS = [
    'Débutant (< 6a)',
    'Intermédiaire bas (6a–6b+)',
    'Intermédiaire (6c–7a+)',
    'Avancé (7b–7c+)',
    'Expert (8a+)'
]
LEVEL_RANGES = [
    {'min_fra': '5a', 'max_fra': '6a'},
    {'min_fra': '6a', 'max_fra': '6b+'},
    {'min_fra': '6c', 'max_fra': '7a+'},
    {'min_fra': '7b', 'max_fra': '7c+'},
    {'min_fra': '8a', 'max_fra': '8c+'},
]

df['level_class'] = pd.cut(
    df['grades_max'],
    bins=LEVEL_BINS,
    labels=[0, 1, 2, 3, 4],
    right=False
).astype(int)

print("Distribution des classes :")
for i, label in enumerate(LEVEL_LABELS):
    n = (df['level_class'] == i).sum()
    print(f"  Classe {i} — {label:30s} : {n:5d} ({100*n/len(df):.1f}%)")

## 3. Entraînement

In [ ]:
FEATURES = ['sex', 'height', 'weight', 'bmi', 'age', 'years_cl']
TARGET = 'level_class'

df_ml = df[df['years_cl'] >= 1].copy()
X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5,
                             class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                                  random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
acc_gb = accuracy_score(y_test, y_pred_gb)

print(f"Random Forest     — Accuracy : {acc_rf:.3f}")
print(f"Gradient Boosting — Accuracy : {acc_gb:.3f}")

best_model = rf if acc_rf >= acc_gb else gb
best_name = 'Random Forest' if acc_rf >= acc_gb else 'Gradient Boosting'
y_pred_best = y_pred_rf if best_name == 'Random Forest' else y_pred_gb
print(f"\nMeilleur modèle : {best_name}")

## 4. Évaluation détaillée

In [ ]:
print(classification_report(y_test, y_pred_best, target_names=LEVEL_LABELS))

In [ ]:
# Accuracy ±1 classe (tolérance d'une classe d'écart)
within_1 = np.mean(np.abs(y_test - y_pred_best) <= 1) * 100
print(f"Accuracy exacte   : {accuracy_score(y_test, y_pred_best)*100:.1f}%")
print(f"Accuracy ±1 classe : {within_1:.1f}%")

# Validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"CV 5-fold : {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best)
short_labels = ['Déb.', 'Int.bas', 'Inter.', 'Avancé', 'Expert']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels, ax=axes[0])
axes[0].set_title(f'Matrice de confusion — {best_name}')
axes[0].set_ylabel('Classe réelle')
axes[0].set_xlabel('Classe prédite')

# Feature importance
importances = best_model.feature_importances_
axes[1].barh(FEATURES, importances, color='steelblue')
axes[1].set_title('Feature importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../data/processed/model_climbing_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Mode POTENTIEL — règles métier débutants absolus

In [ ]:
def predict_climbing_potential(weight_kg, height_cm, age, dead_hang_sec=None, max_pullups=None):
    bmi = weight_kg / (height_cm / 100) ** 2
    score = 0

    if bmi < 20:   score += 3
    elif bmi < 22: score += 2
    elif bmi < 25: score += 1

    if dead_hang_sec is not None:
        if dead_hang_sec >= 60:   score += 3
        elif dead_hang_sec >= 40: score += 2
        elif dead_hang_sec >= 20: score += 1

    if max_pullups is not None:
        if max_pullups >= 15:   score += 3
        elif max_pullups >= 10: score += 2
        elif max_pullups >= 5:  score += 1

    if 20 <= age <= 30: score += 1

    if score <= 2:   level = 0
    elif score <= 5: level = 1
    elif score <= 8: level = 2
    else:            level = 3

    return {
        'mode': 'potential',
        'level_class': level,
        'level_label': LEVEL_LABELS[level],
        'grade_range': f"{LEVEL_RANGES[level]['min_fra']} → {LEVEL_RANGES[level]['max_fra']}",
        'score': score,
        'confidence': 'low',
        'disclaimer': (
            "Estimation basée sur votre profil physique uniquement. "
            "Votre niveau réel dépendra de votre entraînement, technique et régularité. "
            "Cette fourchette représente un potentiel après 1-2 ans de pratique sérieuse."
        )
    }

# Tests
print(predict_climbing_potential(65, 175, 25, dead_hang_sec=50, max_pullups=12))
print(predict_climbing_potential(80, 180, 35, dead_hang_sec=20, max_pullups=5))

## 6. Fonction de prédiction unifiée

In [ ]:
def predict_climbing(model, sex, height, weight, age, years_cl,
                     dead_hang_sec=None, max_pullups=None):
    if years_cl == 0:
        return predict_climbing_potential(weight, height, age, dead_hang_sec, max_pullups)

    bmi = weight / (height / 100) ** 2
    features = pd.DataFrame([[sex, height, weight, bmi, age, years_cl]], columns=FEATURES)

    level_pred = int(model.predict(features)[0])
    proba = model.predict_proba(features)[0]
    confidence = 'high' if proba.max() >= 0.5 else 'medium'

    return {
        'mode': 'ml',
        'level_class': level_pred,
        'level_label': LEVEL_LABELS[level_pred],
        'grade_range': f"{LEVEL_RANGES[level_pred]['min_fra']} → {LEVEL_RANGES[level_pred]['max_fra']}",
        'confidence': confidence,
        'probabilities': {LEVEL_LABELS[i]: round(float(p), 3) for i, p in enumerate(proba)},
        'disclaimer': None
    }

print("Débutant absolu :")
print(predict_climbing(best_model, 0, 175, 65, 25, years_cl=0, dead_hang_sec=45, max_pullups=10))
print()
print("Grimpeur 3 ans :")
print(predict_climbing(best_model, 0, 175, 65, 25, years_cl=3))
print()
print("Grimpeur 10 ans :")
print(predict_climbing(best_model, 0, 170, 62, 30, years_cl=10))

## 7. Export

In [ ]:
joblib.dump(best_model, '../models/climbing_model.pkl')
joblib.dump(grade_map, '../models/climbing_grade_map.pkl')

level_config = {
    'labels': LEVEL_LABELS,
    'ranges': LEVEL_RANGES,
    'bins': LEVEL_BINS
}
joblib.dump(level_config, '../models/climbing_level_config.pkl')

cv_mean = cv_scores.mean()
within_1_val = float(within_1)

metadata = {
    'model_type': best_name,
    'problem_type': 'classification',
    'sport': 'climbing',
    'features': FEATURES,
    'target': 'level_class (0-4)',
    'level_labels': LEVEL_LABELS,
    'level_ranges': LEVEL_RANGES,
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'metrics': {
        'accuracy': round(accuracy_score(y_test, y_pred_best), 4),
        'within_1_class_pct': round(within_1_val, 1),
        'cv_accuracy_mean': round(float(cv_mean), 4),
        'cv_accuracy_std': round(float(cv_scores.std()), 4)
    },
    'prediction_modes': {
        'potential': 'years_cl = 0 → règles physiologiques (dead hang, BMI, tractions)',
        'ml': 'years_cl ≥ 1 → classification Random Forest'
    },
    'why_classification': (
        'La régression sur grade précis donnait MAE=6.7 grades (R²=0.25). '
        'La classification par niveau est plus honnête et plus utile pour l utilisateur.'
    ),
    'dataset': 'jordizar/climb-dataset (8a.nu)',
    'trained_at': datetime.now().isoformat(),
    'limitations': [
        'Dataset surreprésenté en grimpeurs avancés (médiane 7b)',
        'Aucun grimpeur avec 0 an de pratique dans les données',
        'Dead hang absent du dataset — utilisé uniquement en mode potentiel'
    ]
}

with open('../models/climbing_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Modèles et config exportés dans models/")
print()
print(json.dumps(metadata['metrics'], indent=2))